In [7]:
import numpy as np
from qewton.config.axes import BatchAxes, FeatureAxes, GeometryAxes
from qewton.visualization import *
from qewton import DataConfiguration
images = 255 * np.random.rand(20, 128, 128, 2)


from qewton.config import Variable
import qewton

batch_axes = BatchAxes(20)

batch = SliderSpec(
    batch_axes,  
    init_state=0,
    minimum=0,
    maximum=19,
    step=1,
)


X = Variable("x", dim=1)
Y = Variable("y", dim=1)
Z = Variable("z", dim=1)
C = Variable("c", dim=1)

data_config = DataConfiguration(
    batch_axes,
    GeometryAxes(qewton.geometries.Geometry(X*Y, shape=(128, 128))),
    FeatureAxes(Z * C))

plot = SurfacePlot(
    images,
    data_config,
    x = X,
    y = Y,
    z = AxisSpec(Z, log_scale=True),
    color = C,
    controls = [batch]
)

figure = Figure(plot, title="Image Plot")

app = DashApplication.create(figure)

app.run(debug=True, jupyter_mode="external")

Dash app running on http://127.0.0.1:8050/


In [6]:
from qewton.config.variables import Variable
from qewton.geometries.continuous.domains_2d.circle import Circle
from qewton.visualization.plots.geometry import GeometryPlot
from qewton.visualization.figure import Figure
from qewton.visualization import *

x = Variable("x", 2)  # 2D-Variable, wie von Box/Sphere/etc. verlangt

box = Circle(variable=x, center=[0, 0], radius=1.0)
plot = GeometryPlot(box, max_vertex_distance=0.2, show_edges=False)
fig = Figure(plot, title="Circle")


app = DashApplication.create(fig)
app.run(debug=True, jupyter_mode="external")

Dash app running on http://127.0.0.1:8050/


In [2]:
import numpy as np
import qewton
from qewton import DataConfiguration
from qewton.config import Variable
from qewton.config.axes import BatchAxes, FeatureAxes, GeometryAxes
from qewton.geometries.continuous.domains_2d.rectangle import Rectangle
from qewton.geometries.continuous.domains_3d.cylinder import Cylinder
from qewton.geometries.discrete.mesh_geometry import MeshGeometry
from qewton.visualization import *

x3 = Variable("x", 3)
U = Variable("u", 1)          # e.g. temperature, pressure, |velocity|

sphere = Cylinder(variable=x3, center=[0, 0, 0], radius=1.0, height=1.0)
mesh_geometry = sphere.create_mesh(max_vertex_distance=0.15)
vertices = mesh_geometry.mesh.vertices

# One scalar per vertex - here a smooth analytic field for testing
u = np.sin(3 * vertices[:, 0]) * np.cos(3 * vertices[:, 1])
data = u[:, None]                                    # shape (n_vertices, 1)

config = DataConfiguration(
    GeometryAxes(mesh_geometry),
    FeatureAxes(U),
)

plot = MeshFieldPlot(data, config, color=ColorSpec(U, cmap="viridis"), show_edges=False)
fig = Figure(plot, title="Field on sphere surface")

app = DashApplication.create(fig)
app.run(debug=True, jupyter_mode="external")

/tmp/ipykernel_547900/3808651055.py:19: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  u = np.sin(3 * vertices[:, 0]) * np.cos(3 * vertices[:, 1])


Dash app running on http://127.0.0.1:8050/


In [4]:
T = Variable("t", 1)
n_steps, n_vertices = 50, len(mesh_geometry.mesh.vertices)

# e.g. a decaying solution over time, shape (n_steps, n_vertices, 1)
solution = np.random.rand(n_steps, n_vertices, 1) * np.exp(
    -np.linspace(0, 3, n_steps)
)[:, None, None]

time_axes = BatchAxes(n_steps)
config = DataConfiguration(
    time_axes,
    GeometryAxes(mesh_geometry),
    FeatureAxes(T),
)

plot = MeshSurfacePlot(
    solution, config,
    z=T,
    controls=[SliderSpec(time_axes, 0, 0, 50)],   # min/max/init resolved from the config
)

app = DashApplication.create(Figure(plot, title="Time evolution"))
app.run(debug=True, jupyter_mode="external")

ValueError: MeshSurfacePlot supports (2,)D meshes, got a 3D mesh.